# Notebook 01: limpeza dos dados

**Projeto de Análise de Dados e BI** · Tema: redução de faltas em consultas (no-show) em clínicas

**Equipe:** Cauê Franco, Lucio Lima e Daniel Vieira

## Objetivo deste notebook

Ler o arquivo original de consultas, corrigir os problemas dele e gravar a **base tratada** em
`dados/tratados/consultas_tratadas.csv`. Essa base é a **única fonte** do notebook `02_exploracao` e do dashboard.

- **Fonte:** *Medical Appointment No Shows*, do Kaggle (autor Joni Hoppen, licença CC BY-NC-SA 4.0):
  https://www.kaggle.com/datasets/joniarroba/noshowappointments
- **Regra de ouro: o arquivo original não se altera nunca.** O notebook só lê `dados/original/KaggleV2-May-2016.csv`
  e grava o resultado em outro arquivo. Para provar, conferimos o hash SHA-256 do original no começo e no fim.

## Definições usadas no projeto inteiro

- **Taxa de falta** = consultas com `faltou = 1` ÷ total de consultas, em %.
- **Espera** (`dias_espera`) = dias de calendário entre o agendamento e a consulta.

## Os 9 passos

| Passo | O que fazemos |
|---|---|
| 1 | Renomear as colunas |
| 2 | Converter as datas |
| 3 | Remover a idade negativa |
| 4 | Remover as consultas anteriores ao agendamento |
| 5 | Manter as idades 0 e acima de 100 |
| 6 | Manter a coluna de deficiência (0 a 4) |
| 7 | Criar `faltou` (0 ou 1) |
| 8 | Criar as colunas derivadas |
| 9 | Conferência final e gravação |

Cada passo tem o mesmo formato: **Problema**, **Decisão**, **Justificativa**, o código, o antes e depois e um `assert`
(uma checagem que para o notebook se algo estiver errado).

## Preparação: carga e foto do original

In [1]:
import hashlib
from pathlib import Path

import pandas as pd

# Os caminhos são relativos à pasta notebooks/ (onde o notebook roda).
CAMINHO_ORIGINAL = Path("../dados/original/KaggleV2-May-2016.csv")
CAMINHO_TRATADO = Path("../dados/tratados/consultas_tratadas.csv")

# SHA-256 do arquivo original, conferido em 2026-11-17.
HASH_ORIGINAL = "9132d3e7d0246617df9041d3764f20ad6f08e7b0d9f0997fa254fc5e52eda27d"


def sha256_do_arquivo(caminho):
    # Lê o arquivo em blocos e devolve o SHA-256 dele.
    h = hashlib.sha256()
    with open(caminho, "rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()


def fmt(n):
    # Número inteiro no padrão brasileiro: 110527 vira 110.527.
    return f"{n:,}".replace(",", ".")


# Antes de qualquer coisa: o original é mesmo o arquivo esperado?
hash_inicial = sha256_do_arquivo(CAMINHO_ORIGINAL)
print("SHA-256 do original (início):", hash_inicial)
assert hash_inicial == HASH_ORIGINAL, "O arquivo original mudou!"
print("OK: o hash bate com o esperado.")

SHA-256 do original (início): 9132d3e7d0246617df9041d3764f20ad6f08e7b0d9f0997fa254fc5e52eda27d
OK: o hash bate com o esperado.


In [2]:
# PatientId é só um identificador: lemos como texto para não virar número (e perder dígitos ou ganhar notação científica).
original = pd.read_csv(CAMINHO_ORIGINAL, dtype={"PatientId": str})

# 'df' é a cópia que os passos seguintes transformam; 'original' nunca é modificado.
df = original.copy()

In [3]:
# Foto do original
print("Formato (linhas, colunas):", original.shape)
print()
print("Tipos de cada coluna:")
print(original.dtypes.to_string())
print()
print("Valores nulos por coluna:")
print(original.isna().sum().to_string())
print()
print("Linhas duplicadas:", original.duplicated().sum())
print()
print("Idade (describe):")
print(original["Age"].describe().round(1).to_string())

Formato (linhas, colunas): (110527, 14)

Tipos de cada coluna:
PatientId           str
AppointmentID     int64
Gender              str
ScheduledDay        str
AppointmentDay      str
Age               int64
Neighbourhood       str
Scholarship       int64
Hipertension      int64
Diabetes          int64
Alcoholism        int64
Handcap           int64
SMS_received      int64
No-show             str

Valores nulos por coluna:
PatientId         0
AppointmentID     0
Gender            0
ScheduledDay      0
AppointmentDay    0
Age               0
Neighbourhood     0
Scholarship       0
Hipertension      0
Diabetes          0
Alcoholism        0
Handcap           0
SMS_received      0
No-show           0



Linhas duplicadas: 0

Idade (describe):
count    110527.0
mean         37.1
std          23.1
min          -1.0
25%          18.0
50%          37.0
75%          55.0
max         115.0


In [4]:
# Asserts da carga
assert original.shape == (110527, 14), original.shape
assert original.isna().sum().sum() == 0, "O original tinha nulos"
assert original.duplicated().sum() == 0, "O original tinha linhas duplicadas"
print(f"OK: {fmt(len(original))} linhas, {original.shape[1]} colunas, zero nulos, zero duplicatas.")

OK: 110.527 linhas, 14 colunas, zero nulos, zero duplicatas.


**O que a foto mostra:** 110.527 consultas e 14 colunas, sem nulos e sem linhas repetidas. Os problemas estão
escondidos nos valores e nos nomes, e é isso que os passos a seguir tratam. Nota: cinco `PatientId` vêm escritos com
`E-5` no original (por exemplo `9377952927E-5`). Como o identificador é só um rótulo, ele é gravado como texto, igual
ao original, sem tratamento.

## Passo 1: nomes das colunas

**Problema.** Alguns nomes estão com a grafia errada (`Hipertension`, `Handcap`), um tem hífen (`No-show`) e todos
estão em inglês, com estilos misturados (`PatientId`, `SMS_received`).

**Decisão.** Renomear as 14 colunas para português, em minúsculas e com `_` (snake_case), pelo mapa abaixo.

**Justificativa.** Nomes corretos e padronizados evitam erro de digitação. O hífen de `No-show` atrapalha em fórmulas
e no Power BI. Em português, o time e o professor leem as colunas sem tradução.

In [5]:
MAPA_COLUNAS = {
    "PatientId": "paciente_id",
    "AppointmentID": "consulta_id",
    "Gender": "sexo",
    "ScheduledDay": "data_agendamento",
    "AppointmentDay": "data_consulta",
    "Age": "idade",
    "Neighbourhood": "bairro",
    "Scholarship": "bolsa_familia",
    "Hipertension": "hipertensao",
    "Diabetes": "diabetes",
    "Alcoholism": "alcoolismo",
    "Handcap": "deficiencia",
    "SMS_received": "recebeu_sms",
    "No-show": "faltou",
}

colunas_antes = list(df.columns)
df = df.rename(columns=MAPA_COLUNAS)
colunas_depois = list(df.columns)

In [6]:
# Antes e depois
pd.DataFrame({"antes (original)": colunas_antes, "depois (tratada)": colunas_depois})

,antes (original),depois (tratada)
0,PatientId,paciente_id
1,AppointmentID,consulta_id
2,Gender,sexo
3,ScheduledDay,data_agendamento
4,AppointmentDay,data_consulta
5,Age,idade
6,Neighbourhood,bairro
7,Scholarship,bolsa_familia
8,Hipertension,hipertensao
9,Diabetes,diabetes


In [7]:
# Assert do passo 1
assert colunas_antes == list(MAPA_COLUNAS.keys()), "O mapa não cobre as colunas do original"
assert colunas_depois == list(MAPA_COLUNAS.values()), colunas_depois
assert len(colunas_depois) == 14
print("OK: 14 colunas renomeadas, na mesma ordem do original.")

OK: 14 colunas renomeadas, na mesma ordem do original.


## Passo 2: datas

**Problema.** `ScheduledDay` e `AppointmentDay` são texto (por exemplo `2016-04-29T18:38:08Z`). O agendamento tem hora,
mas a consulta vem sempre com `00:00:00`: a base não diz o horário da consulta.

**Decisão.** Converter as duas colunas para **data** (`AAAA-MM-DD`), descartando a hora e o fuso. A espera passa a ser
contada em **dias de calendário**: `data_consulta − data_agendamento`.

**Justificativa.**
- Como a consulta não tem horário, a espera só pode ser contada em dias. Por isso a hora do agendamento é descartada.
- O `Z` no fim do texto diz "horário UTC", mas os agendamentos acontecem entre 6h e 21h, com pico às 7h (veja abaixo).
  Isso é horário de funcionamento de clínica, ou seja, tudo indica que é o **horário local rotulado errado como UTC**.
  Por isso **não convertemos o fuso**: pegamos a data como está escrita, que é o dia que a clínica e o paciente
  viram. Converter deslocaria horários e faria algumas datas mudarem de dia sem motivo.

In [8]:
# Evidência do fuso: a hora do agendamento (lida do texto, sem converter nada)
hora_agendamento = original["ScheduledDay"].str[11:13].astype(int)
print("Hora do agendamento: mínima", hora_agendamento.min(), "| máxima", hora_agendamento.max())
print("Horas mais frequentes (hora: consultas):")
print(hora_agendamento.value_counts().head(3).to_string())
print()
print("AppointmentDay sempre sem hora?", bool((original["AppointmentDay"].str[11:] == "00:00:00Z").all()))

Hora do agendamento: mínima 6 | máxima 21
Horas mais frequentes (hora: consultas):
ScheduledDay
7    19213
8    15349
9    12823

AppointmentDay sempre sem hora? True


In [9]:
# Antes: texto como veio
datas_antes = df[["data_agendamento", "data_consulta"]].head(3)

# Pega só os 10 primeiros caracteres (AAAA-MM-DD): descarta hora e fuso sem converter.
df["data_agendamento"] = pd.to_datetime(df["data_agendamento"].str[:10], format="%Y-%m-%d")
df["data_consulta"] = pd.to_datetime(df["data_consulta"].str[:10], format="%Y-%m-%d")

In [10]:
# Antes e depois
print("ANTES (texto):")
print(datas_antes.to_string())
print()
print("DEPOIS (data):")
print(df[["data_agendamento", "data_consulta"]].head(3).to_string())
print()
print(df[["data_agendamento", "data_consulta"]].dtypes.to_string())
print()
print("Agendamentos de", df["data_agendamento"].min().date(), "a", df["data_agendamento"].max().date())
print("Consultas de   ", df["data_consulta"].min().date(), "a", df["data_consulta"].max().date())

ANTES (texto):
       data_agendamento         data_consulta
0  2016-04-29T18:38:08Z  2016-04-29T00:00:00Z
1  2016-04-29T16:08:27Z  2016-04-29T00:00:00Z
2  2016-04-29T16:19:04Z  2016-04-29T00:00:00Z

DEPOIS (data):
  data_agendamento data_consulta
0       2016-04-29    2016-04-29
1       2016-04-29    2016-04-29
2       2016-04-29    2016-04-29

data_agendamento    datetime64[us]
data_consulta       datetime64[us]

Agendamentos de 2015-11-10 a 2016-06-08
Consultas de    2016-04-29 a 2016-06-08


In [11]:
# Assert do passo 2
for col in ["data_agendamento", "data_consulta"]:
    assert pd.api.types.is_datetime64_any_dtype(df[col]), col
    assert df[col].notna().all(), f"{col} ficou com data nula na conversão"
    assert (df[col] == df[col].dt.normalize()).all(), f"{col} ainda tem hora"
print("OK: as duas colunas são datas, sem hora e sem nulos.")

OK: as duas colunas são datas, sem hora e sem nulos.


## Passo 3: idade negativa

**Problema.** Há 1 consulta com idade **-1**, o que é impossível.

**Decisão.** Remover essa linha.

**Justificativa.** É 1 linha em 110.527 e não há como descobrir a idade verdadeira. Chutar um valor seria inventar
dado. Remover não muda nenhuma taxa de forma perceptível.

In [12]:
# Registro das remoções: cada regra de remoção entra aqui e o passo 9 monta a tabela-resumo.
REMOCOES = []

linhas_antes_p3 = len(df)
idade_minima_antes = df["idade"].min()
mask_idade_negativa = df["idade"] < 0

print("Linha com idade negativa (antes de remover):")
print(df.loc[mask_idade_negativa].T.to_string())

REMOCOES.append({"passo": 3, "regra": "Idade negativa (-1)", "linhas_removidas": int(mask_idade_negativa.sum())})
df = df.loc[~mask_idade_negativa].reset_index(drop=True)

Linha com idade negativa (antes de remover):
                                99832
paciente_id           465943158731293
consulta_id                   5775010
sexo                                F
data_agendamento  2016-06-06 00:00:00
data_consulta     2016-06-06 00:00:00
idade                              -1
bairro                          ROMÃO
bolsa_familia                       0
hipertensao                         0
diabetes                            0
alcoolismo                          0
deficiencia                         0
recebeu_sms                         0
faltou                             No


In [13]:
# Antes e depois
print(f"Linhas: {fmt(linhas_antes_p3)} -> {fmt(len(df))}")
print(f"Idade mínima: {idade_minima_antes} -> {df['idade'].min()}")

Linhas: 110.527 -> 110.526
Idade mínima: -1 -> 0


In [14]:
# Assert do passo 3
assert (df["idade"] >= 0).all(), "Ainda há idade negativa"
assert len(df) == linhas_antes_p3 - 1
assert REMOCOES[-1]["linhas_removidas"] == 1
print("OK: nenhuma idade negativa e 1 linha a menos.")

OK: nenhuma idade negativa e 1 linha a menos.


## Passo 4: consultas antes do agendamento

**Problema.** 5 consultas têm data **anterior** ao dia em que foram agendadas (de -6 a -1 dia de diferença). Não dá
para ir à consulta antes de marcá-la.

**Decisão.** Remover as 5 linhas.

**Justificativa.** São impossíveis, e não temos como saber qual das duas datas está errada, então não dá para
corrigir. Se ficassem, teriam **espera negativa**, e a espera é justamente a variável central da nossa análise.

In [15]:
linhas_antes_p4 = len(df)
espera_temporaria = (df["data_consulta"] - df["data_agendamento"]).dt.days
mask_antes_agendamento = espera_temporaria < 0

print("As consultas com data anterior ao agendamento (antes de remover):")
inconsistentes = df.loc[mask_antes_agendamento, ["consulta_id", "data_agendamento", "data_consulta"]].copy()
inconsistentes["diferenca_em_dias"] = espera_temporaria[mask_antes_agendamento]
print(inconsistentes.to_string(index=False))

REMOCOES.append({
    "passo": 4,
    "regra": "Consulta antes do agendamento",
    "linhas_removidas": int(mask_antes_agendamento.sum()),
})
df = df.loc[~mask_antes_agendamento].reset_index(drop=True)

As consultas com data anterior ao agendamento (antes de remover):
 consulta_id data_agendamento data_consulta  diferenca_em_dias
     5679978       2016-05-10    2016-05-09                 -1
     5715660       2016-05-18    2016-05-17                 -1
     5664962       2016-05-05    2016-05-04                 -1
     5686628       2016-05-11    2016-05-05                 -6
     5655637       2016-05-04    2016-05-03                 -1


In [16]:
# Antes e depois
espera_depois = (df["data_consulta"] - df["data_agendamento"]).dt.days
print(f"Linhas: {fmt(linhas_antes_p4)} -> {fmt(len(df))}")
print(f"Menor diferença entre as datas (dias): {espera_temporaria.min()} -> {espera_depois.min()}")

Linhas: 110.526 -> 110.521
Menor diferença entre as datas (dias): -6 -> 0


In [17]:
# Assert do passo 4
assert (espera_depois >= 0).all(), "Ainda há consulta antes do agendamento"
assert len(df) == linhas_antes_p4 - 5
assert REMOCOES[-1]["linhas_removidas"] == 5
print("OK: nenhuma consulta antes do agendamento e 5 linhas a menos.")

OK: nenhuma consulta antes do agendamento e 5 linhas a menos.


## Passo 5: idade 0 e idade acima de 100 (ficam)

**Problema.** Duas faixas de idade parecem estranhas: **3.539** consultas com idade **0** e **7** com idade acima de
100 (5 com 115 anos e 2 com 102).

**Decisão.** Manter as duas. Nenhuma linha é removida neste passo.

**Justificativa.**
- Idade 0 são bebês com menos de 1 ano, o que é perfeitamente possível numa consulta.
- Acima de 100 é raro, mas possível, e não temos prova de que seja erro. São só 7 linhas em mais de 110 mil, então não
  mudam as taxas. Elas caem na faixa `60+`.

In [18]:
linhas_antes_p5 = len(df)
idade_zero = int((df["idade"] == 0).sum())
acima_de_100 = df.loc[df["idade"] > 100, "idade"]

print("Consultas com idade 0:", fmt(idade_zero))
print("Consultas com idade acima de 100:", len(acima_de_100))
print(acima_de_100.value_counts().rename_axis("idade").rename("consultas").to_string())

Consultas com idade 0: 3.539
Consultas com idade acima de 100: 7
idade
115    5
102    2


In [19]:
# Antes e depois (nada muda neste passo)
print(f"Linhas: {fmt(linhas_antes_p5)} -> {fmt(len(df))}")

Linhas: 110.521 -> 110.521


In [20]:
# Assert do passo 5
assert idade_zero == 3539, idade_zero
assert len(acima_de_100) == 7, len(acima_de_100)
assert sorted(acima_de_100.unique()) == [102, 115]
assert len(df) == linhas_antes_p5
print("OK: 3.539 com idade 0 e 7 acima de 100, todas mantidas.")

OK: 3.539 com idade 0 e 7 acima de 100, todas mantidas.


## Passo 6: coluna `deficiencia` (0 a 4)

**Problema.** A fonte descreve essa coluna (`Handcap`) como verdadeiro/falso, mas ela vai de **0 a 4**, e a fonte não
explica o que 2, 3 e 4 significam (talvez o número de deficiências, talvez o tipo).

**Decisão.** Manter o valor como está, de 0 a 4, e registrar a dúvida. No Passo 8 criamos também `tem_deficiencia`
(0 ou 1).

**Justificativa.** Não dá para "corrigir" o que não sabemos. Manter o valor não perde informação, e a versão 0/1 usa só
o que é seguro afirmar: acima de 0 significa que a pessoa tem alguma deficiência registrada.

In [21]:
contagem_original = original["Handcap"].value_counts().sort_index()
contagem_atual = df["deficiencia"].value_counts().sort_index()
pd.DataFrame({
    "no original (110.527 linhas)": contagem_original,
    "após os passos 3 e 4": contagem_atual,
}).rename_axis("deficiencia")

,no original (110.527 linhas),após os passos 3 e 4
deficiencia,,
0,108286,108282
1,2042,2040
2,183,183
3,13,13
4,3,3


In [22]:
# Assert do passo 6
assert sorted(df["deficiencia"].unique()) == [0, 1, 2, 3, 4]
assert contagem_original.to_dict() == {0: 108286, 1: 2042, 2: 183, 3: 13, 4: 3}, contagem_original.to_dict()
print("OK: a coluna vai de 0 a 4, como no original, e nenhum valor foi alterado.")

OK: a coluna vai de 0 a 4, como no original, e nenhum valor foi alterado.


## Passo 7: `faltou` (0 ou 1)

**Problema.** A coluna `No-show` guarda `Yes` (faltou) e `No` (compareceu). O nome tem um "No" duplo que confunde:
`No-show = No` quer dizer "não faltou".

**Decisão.** Criar `faltou`: **1** para `Yes` (faltou) e **0** para `No` (compareceu).

**Justificativa.** Com 0 e 1, a **média** da coluna já é a taxa de falta, e a soma é o número de faltas. O nome
`faltou = 1` se lê direto, sem dupla negação.

In [23]:
contagem_antes_faltou = df["faltou"].value_counts()
faltas_yes = int((df["faltou"] == "Yes").sum())

mapa_faltou = {"Yes": 1, "No": 0}
df["faltou"] = df["faltou"].map(mapa_faltou)
assert df["faltou"].notna().all(), "Apareceu um valor diferente de Yes/No"
df["faltou"] = df["faltou"].astype(int)

In [24]:
# Antes e depois
print("ANTES (Yes/No):")
print(contagem_antes_faltou.to_string())
print()
print("DEPOIS (0/1):")
print(df["faltou"].value_counts().sort_index().to_string())
print()
taxa = f"{df['faltou'].mean() * 100:.1f}".replace(".", ",")
print(f"Taxa de falta nas {fmt(len(df))} linhas: {taxa}%")

ANTES (Yes/No):
faltou
No     88207
Yes    22314

DEPOIS (0/1):
faltou
0    88207
1    22314

Taxa de falta nas 110.521 linhas: 20,2%


In [25]:
# Assert do passo 7
assert set(df["faltou"].unique()) == {0, 1}
assert df["faltou"].sum() == faltas_yes, "A soma de faltou não bate com a contagem de Yes"
print(f"OK: faltou só tem 0 e 1, e a soma ({fmt(int(df['faltou'].sum()))}) bate com os Yes.")

OK: faltou só tem 0 e 1, e a soma (22.314) bate com os Yes.


## Passo 8: colunas derivadas

**Problema.** Para analisar a falta por espera, por idade e por dia da semana, falta transformar as colunas em grupos
prontos.

**Decisão.** Criar 5 colunas a partir das que já existem:
- `dias_espera`: `data_consulta − data_agendamento`, em dias.
- `dia_semana`: o dia da semana da consulta (`Segunda` a `Sábado`).
- `faixa_etaria`: `0-12`, `13-17`, `18-29`, `30-44`, `45-59`, `60+`.
- `faixa_espera`: `Mesmo dia` (0 dias), `1-7 dias`, `8-15 dias`, `16-30 dias`, `31+ dias`.
- `tem_deficiencia`: 1 se `deficiencia` for maior que 0.

**Justificativa.** As faixas têm cortes fechados e rótulos fixos, que valem **igual** no notebook 02, no dashboard e
nos slides. O dia da semana é calculado pelo número do dia, para não depender do idioma do computador. Os rótulos usam
hífen simples (`-`) para não dar problema no Power BI.

In [26]:
ORDEM_DIA_SEMANA = ["Segunda", "Terça", "Quarta", "Quinta", "Sexta", "Sábado"]
ORDEM_FAIXA_ETARIA = ["0-12", "13-17", "18-29", "30-44", "45-59", "60+"]
ORDEM_FAIXA_ESPERA = ["Mesmo dia", "1-7 dias", "8-15 dias", "16-30 dias", "31+ dias"]

# 0 = segunda ... 5 = sábado. O domingo (6) não existe nos dados: se existisse, o assert abaixo avisa.
NOMES_DIA = dict(enumerate(ORDEM_DIA_SEMANA))

# Espera em dias de calendário
df["dias_espera"] = (df["data_consulta"] - df["data_agendamento"]).dt.days

# Dia da semana da consulta, como categoria ordenada
df["dia_semana"] = pd.Categorical(
    df["data_consulta"].dt.dayofweek.map(NOMES_DIA),
    categories=ORDEM_DIA_SEMANA,
    ordered=True,
)

# Faixa etária: (-1, 12] = 0 a 12, (12, 17] = 13 a 17, e assim por diante.
df["faixa_etaria"] = pd.cut(
    df["idade"],
    bins=[-1, 12, 17, 29, 44, 59, float("inf")],
    labels=ORDEM_FAIXA_ETARIA,
    ordered=True,
)

# Faixa de espera: (-1, 0] = mesmo dia, (0, 7] = 1 a 7 dias, e assim por diante.
df["faixa_espera"] = pd.cut(
    df["dias_espera"],
    bins=[-1, 0, 7, 15, 30, float("inf")],
    labels=ORDEM_FAIXA_ESPERA,
    ordered=True,
)

df["tem_deficiencia"] = (df["deficiencia"] > 0).astype(int)

In [27]:
# Antes e depois: quantas consultas caíram em cada grupo
print("Colunas:", len(df.columns), "(eram 14 antes deste passo)")
print()
for coluna in ["dia_semana", "faixa_etaria", "faixa_espera"]:
    print(df[coluna].value_counts(sort=False).rename_axis(coluna).rename("consultas").to_string())
    print()
print("tem_deficiencia:")
print(df["tem_deficiencia"].value_counts().sort_index().to_string())
print()
print("dias_espera: mínimo", df["dias_espera"].min(), "| máximo", df["dias_espera"].max())
print("Exemplo de linhas:")
print(df[["data_agendamento", "data_consulta", "dias_espera", "dia_semana", "idade", "faixa_etaria", "faixa_espera"]].head(3).to_string(index=False))

Colunas: 19 (eram 14 antes deste passo)

dia_semana
Segunda    22713
Terça      25638
Quarta     25866
Quinta     17246
Sexta      19019
Sábado        39

faixa_etaria
0-12     21035
13-17     6343
18-29    16729
30-44    22021
45-59    23221
60+      21172

faixa_espera
Mesmo dia     38562
1-7 dias      32185
8-15 dias     13528
16-30 dias    15868
31+ dias      10378

tem_deficiencia:
tem_deficiencia
0    108282
1      2239

dias_espera: mínimo 0 | máximo 179
Exemplo de linhas:
data_agendamento data_consulta  dias_espera dia_semana  idade faixa_etaria faixa_espera
      2016-04-29    2016-04-29            0      Sexta     62          60+    Mesmo dia
      2016-04-29    2016-04-29            0      Sexta     56        45-59    Mesmo dia
      2016-04-29    2016-04-29            0      Sexta     62          60+    Mesmo dia


In [28]:
# Assert do passo 8
assert (df["dias_espera"] >= 0).all()
assert df["tem_deficiencia"].isin([0, 1]).all()
assert ((df["tem_deficiencia"] == 1) == (df["deficiencia"] > 0)).all()

for coluna, ordem in [
    ("dia_semana", ORDEM_DIA_SEMANA),
    ("faixa_etaria", ORDEM_FAIXA_ETARIA),
    ("faixa_espera", ORDEM_FAIXA_ESPERA),
]:
    assert df[coluna].notna().all(), f"{coluna}: alguma linha ficou fora das categorias"
    assert set(df[coluna].astype(str)) <= set(ordem), f"{coluna}: rótulo fora da lista"
    assert df[coluna].value_counts().sum() == len(df), f"{coluna}: a soma das faixas não fecha"

assert ((df["faixa_espera"] == "Mesmo dia") == (df["dias_espera"] == 0)).all()
print("OK: sem faixas nulas, rótulos iguais aos da lista e 'Mesmo dia' = 0 dias de espera.")

OK: sem faixas nulas, rótulos iguais aos da lista e 'Mesmo dia' = 0 dias de espera.


**Observação (nada é alterado):** deixamos como estão os **sábados** (poucas consultas) e os **bairros com poucas
consultas**. A cautela vai para a leitura da análise: taxas de grupos muito pequenos oscilam muito e não devem virar
conclusão.

In [29]:
consultas_por_bairro = df["bairro"].value_counts()
print("Consultas aos sábados:", int((df["dia_semana"] == "Sábado").sum()))
print("Bairros com menos de 50 consultas:", int((consultas_por_bairro < 50).sum()), "de", len(consultas_por_bairro))

Consultas aos sábados: 39
Bairros com menos de 50 consultas: 5 de 81


## Passo 9: conferência final e gravação

**Problema.** Antes de entregar a base, precisamos ter certeza de que ela está certa e de que o original não foi
tocado.

**Decisão.** Reordenar as colunas, rodar as checagens finais, mostrar a tabela-resumo do que entrou e do que saiu,
gravar o CSV, reler o arquivo gravado e conferir de novo o hash do original.

**Justificativa.** Quem usar a base depois (notebook 02 e dashboard) confia que ela já está limpa. Reler o arquivo
gravado garante que o que está no disco é igual ao que tínhamos na memória (datas em `AAAA-MM-DD`, identificadores
sem virar número).

In [30]:
ORDEM_COLUNAS = [
    "paciente_id", "consulta_id", "sexo", "data_agendamento", "data_consulta", "idade", "bairro",
    "bolsa_familia", "hipertensao", "diabetes", "alcoolismo", "recebeu_sms", "deficiencia",
    "faltou", "dias_espera", "dia_semana", "faixa_etaria", "faixa_espera", "tem_deficiencia",
]

df = df[ORDEM_COLUNAS]

# Asserts finais
total_removidas = sum(item["linhas_removidas"] for item in REMOCOES)
assert len(df) == 110521 == len(original) - total_removidas
assert df.isna().sum().sum() == 0, "Há nulos"
assert df.duplicated().sum() == 0, "Há linhas duplicadas"
assert df["consulta_id"].is_unique, "consulta_id repetido"
assert (df["dias_espera"] >= 0).all()
assert (df["idade"] >= 0).all()
assert set(df["faltou"].unique()) == {0, 1}
assert list(df.columns) == ORDEM_COLUNAS
print(f"OK: {fmt(len(df))} linhas, {df.shape[1]} colunas, sem nulos, sem duplicatas.")

OK: 110.521 linhas, 19 colunas, sem nulos, sem duplicatas.


In [31]:
# Tabela-resumo: o que entrou, o que saiu e por quê
linhas_resumo = [{"item": "Consultas no original", "linhas": len(original)}]
for item in REMOCOES:
    linhas_resumo.append({"item": f"Removidas no passo {item['passo']}: {item['regra']}", "linhas": -item["linhas_removidas"]})
linhas_resumo.append({"item": "Consultas na base tratada", "linhas": len(df)})

resumo = pd.DataFrame(linhas_resumo)
resumo["linhas"] = resumo["linhas"].map(fmt)
print(resumo.to_string(index=False))
print()
print(f"Colunas: {original.shape[1]} no original -> {df.shape[1]} na base tratada (5 colunas derivadas criadas no Passo 8)")
print(f"Faltas: {fmt(int(original['No-show'].eq('Yes').sum()))} no original -> {fmt(int(df['faltou'].sum()))} na base tratada")

                                               item  linhas
                              Consultas no original 110.527
          Removidas no passo 3: Idade negativa (-1)      -1
Removidas no passo 4: Consulta antes do agendamento      -5
                          Consultas na base tratada 110.521

Colunas: 14 no original -> 19 na base tratada (5 colunas derivadas criadas no Passo 8)
Faltas: 22.319 no original -> 22.314 na base tratada


In [32]:
# Gravação: datas em AAAA-MM-DD, sem coluna de índice, UTF-8.
CAMINHO_TRATADO.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CAMINHO_TRATADO, index=False, encoding="utf-8", date_format="%Y-%m-%d", lineterminator="\n")
print("Gravado em dados/tratados/consultas_tratadas.csv")

Gravado em dados/tratados/consultas_tratadas.csv


In [33]:
# Releitura do arquivo gravado (paciente_id e as datas voltam como texto, para conferir o que está escrito no disco)
lido = pd.read_csv(CAMINHO_TRATADO, dtype={"paciente_id": str, "data_agendamento": str, "data_consulta": str})

print("Linhas e colunas lidas:", lido.shape)
print("Primeiras linhas do arquivo gravado:")
print(lido.head(3).to_string(index=False))

Linhas e colunas lidas: (110521, 19)
Primeiras linhas do arquivo gravado:
    paciente_id  consulta_id sexo data_agendamento data_consulta  idade          bairro  bolsa_familia  hipertensao  diabetes  alcoolismo  recebeu_sms  deficiencia  faltou  dias_espera dia_semana faixa_etaria faixa_espera  tem_deficiencia
 29872499824296      5642903    F       2016-04-29    2016-04-29     62 JARDIM DA PENHA              0            1         0           0            0            0       0            0      Sexta          60+    Mesmo dia                0
558997776694438      5642503    M       2016-04-29    2016-04-29     56 JARDIM DA PENHA              0            0         0           0            0            0       0            0      Sexta        45-59    Mesmo dia                0
  4262962299951      5642549    F       2016-04-29    2016-04-29     62   MATA DA PRAIA              0            0         0           0            0            0       0            0      Sexta          60+ 

In [34]:
# Asserts da releitura
assert len(lido) == 110521
assert list(lido.columns) == ORDEM_COLUNAS

# paciente_id: igual ao original, como texto. Nenhum ID virou notação científica por arredondamento ('E+').
assert lido["paciente_id"].tolist() == df["paciente_id"].tolist()
assert not lido["paciente_id"].str.contains("E+", regex=False).any(), "ID em notação científica (E+)"
ids_fora_do_padrao = set(lido.loc[~lido["paciente_id"].str.fullmatch(r"\d+"), "paciente_id"])
ids_fora_do_padrao_original = set(original.loc[~original["PatientId"].str.fullmatch(r"\d+"), "PatientId"])
assert ids_fora_do_padrao == ids_fora_do_padrao_original and len(ids_fora_do_padrao) == 5
print("IDs que já vinham com 'E-5' no original (mantidos como estão):", sorted(ids_fora_do_padrao))

# Datas gravadas como AAAA-MM-DD e coerentes com dias_espera
for col in ["data_agendamento", "data_consulta"]:
    assert lido[col].str.fullmatch(r"\d{4}-\d{2}-\d{2}").all(), f"{col} fora do formato AAAA-MM-DD"
espera_relida = (pd.to_datetime(lido["data_consulta"]) - pd.to_datetime(lido["data_agendamento"])).dt.days
assert (espera_relida == lido["dias_espera"]).all()

# Colunas numéricas voltam como inteiros, com os mesmos valores
colunas_inteiras = [
    "consulta_id", "idade", "bolsa_familia", "hipertensao", "diabetes", "alcoolismo", "recebeu_sms",
    "deficiencia", "faltou", "dias_espera", "tem_deficiencia",
]
for col in colunas_inteiras:
    assert pd.api.types.is_integer_dtype(lido[col]), col
    assert (lido[col] == df[col]).all(), col

# Rótulos gravados iguais aos da lista (com hífen simples)
assert set(lido["dia_semana"]) <= set(ORDEM_DIA_SEMANA)
assert set(lido["faixa_etaria"]) <= set(ORDEM_FAIXA_ETARIA)
assert set(lido["faixa_espera"]) <= set(ORDEM_FAIXA_ESPERA)
assert set(lido["sexo"]) == {"F", "M"}
assert lido["faltou"].sum() == df["faltou"].sum()
print("OK: o arquivo gravado tem 110.521 linhas, as 19 colunas na ordem certa, datas em AAAA-MM-DD e IDs como texto.")

IDs que já vinham com 'E-5' no original (mantidos como estão): ['14172416655E-5', '3921784439E-5', '4374175652E-5', '53761528476E-5', '9377952927E-5']


OK: o arquivo gravado tem 110.521 linhas, as 19 colunas na ordem certa, datas em AAAA-MM-DD e IDs como texto.


In [35]:
# O original continua intocado: mesmo hash do começo e mesmas 110.527 linhas.
hash_final = sha256_do_arquivo(CAMINHO_ORIGINAL)
print("SHA-256 do original (fim):", hash_final)
assert hash_final == HASH_ORIGINAL == hash_inicial, "O original foi alterado!"
assert len(pd.read_csv(CAMINHO_ORIGINAL, usecols=["AppointmentID"])) == 110527
print("OK: o original tem o mesmo hash de antes e continua com 110.527 linhas.")

SHA-256 do original (fim): 9132d3e7d0246617df9041d3764f20ad6f08e7b0d9f0997fa254fc5e52eda27d


OK: o original tem o mesmo hash de antes e continua com 110.527 linhas.

## Resumo da limpeza

Partimos de **110.527 consultas** e 14 colunas, sem nulos e sem duplicatas, mas com nomes ruins, datas em texto e alguns
valores impossíveis. Renomeamos as colunas, convertemos as datas (sem mexer no fuso), removemos **6 linhas** (1 com idade
negativa e 5 com consulta antes do agendamento) e mantivemos o que é raro, mas possível (idade 0, idade acima de 100 e
deficiência de 0 a 4). Criamos `faltou` (0/1) e 5 colunas derivadas, chegando a **110.521 linhas e 19 colunas**
em `dados/tratados/consultas_tratadas.csv`, a única base da análise e do dashboard. O arquivo original continua
intocado, com o mesmo hash do começo ao fim.